# 00. 개요 — KIT GPU 서버 통합 관리 사이트

학과 캡스톤용 GPU 서버 여러 대를 **하나의 사이트**에서 관리하는 시스템을 처음부터 구축하는 안내서입니다.
이 안내서를 `00 → 08` 순서대로 따라 하면 같은 환경을 만들 수 있습니다.

## 무엇을 만드나
- 학생 팀은 가입한 뒤 **서버 1대를 2달 동안 독점**해서 씁니다(리눅스 계정 자동 생성, SSH 키 등록, 웹 터미널, 폴더 백업, LLM 키, 캡스톤 DB).
- 서버 관리자는 모든 서버를 **모니터링**하고 GPU 가 멈추면 **강제 재시작**합니다. 서버 추가·팀 관리·감사 로그도 합니다.
- 서버 관리자 가입은 **학과장 승인**이 필요합니다.
- 서버는 **사설 IP** 로 옮기고, 학교 밖에서는 **Tailscale Funnel**(HTTPS 만 공개)로 사이트에 접속합니다.

## 구조
![구조도](img/architecture.svg)

| 구성 | 위치 | 역할 |
|---|---|---|
| K3s (관리 서버 1대) | 관리 서버 | 사이트·DB·LiteLLM·ArgoCD 가 도는 쿠버네티스 |
| `ssm-api` / `ssm-worker` / `ssm-web` | K3s | 백엔드 API, 주기 작업(만료·지표), 화면 |
| `postgres-mgmt` / `postgres-capstone` | K3s | 사이트 데이터, 학생용 캡스톤 DB |
| LiteLLM | K3s | 학생용 LLM API 게이트웨이(제공사는 나중에 연결) |
| ArgoCD | K3s | GitHub 저장소를 보고 자동 배포 |
| Tailscale Funnel | 관리 서버 | 학교 밖에서 사이트 HTTPS 접속 |
| `ssm-agent` | **모든** 서버(호스트) | 계정·키·백업·지표·재부팅 등 허용된 작업만 수행 |

> 학생 서버는 학생에게 sudo 를 주기 때문에 **K3s 에 넣지 않습니다**(노드 인증서를 빼내 클러스터를 공격할 수 있음). 호스트에 에이전트만 설치합니다.

## 흐름
![배포 흐름](img/pipeline.svg)
![서버 가입 순서](img/join.svg)
![네트워크](img/network.svg)

## 역할
| 역할 | 가입 | 할 수 있는 일 |
|---|---|---|
| 학생(팀) | 사이트에서 즉시 | 서버 선택(1대), 웹 터미널, SSH 키, 폴더 백업, LLM 키·DB 신청 |
| 서버 관리자 | 사이트에서 가입 → **학과장 승인** | 전체 모니터링, 서버 추가, 팀 관리, 강제 재시작, 감사 로그 |
| 학과장 | 최초 1회 명령으로 생성 | 관리자 승인·거절·회수 |

## 준비물
- 서버 2대 이상(Ubuntu 24.04), 관리 서버는 **서버실 콘솔**(모니터·키보드)에서 작업할 수 있어야 함
- 학교(통신 담당)에서 받을 값: 사설 대역의 **IP, 마스크, 게이트웨이, DNS**, 사설 대역에서 **인터넷(NAT)** 가능 여부
- GitHub 계정(비공개 저장소), Tailscale 계정(무료)
- 인증 앱(Google/Microsoft Authenticator)이 깔린 휴대폰

## 자리표시자 표
| 자리표시자 | 뜻 | 예 |
|---|---|---|
| `[MGMT_NAME]` | 관리 서버 이름 | `ubuntu01` |
| `[MGMT_PRIVATE_IP]` | 관리 서버 사설 IP | `[대역].[번호]` |
| `[GATEWAY]` | 게이트웨이(학교 안내: 할당 IP 앞 3자리 + `.1`) | |
| `[PREFIX]` | 서브넷 길이 | `24` |
| `[IFACE]` | 랜 인터페이스 이름(`ip -br link` 로 확인) | 서버마다 다름 |
| `[SCHOOL_DNS1]`, `[SCHOOL_DNS2]` | 학교 DNS 두 개 | |
| `[SSH_PORT]` | 서버 SSH 포트 | `7540` (이 구축에서는 22 가 아님) |
| `[TAILNET]` | Tailscale 계정 이름(`tailXXXXXX`) | |
| `[FUNNEL_URL]` | 사이트 주소 `https://[MGMT_NAME].[TAILNET].ts.net` | |
| `[GITHUB_USER]`, `[REPO]` | GitHub 계정, 저장소 이름 | |
| `[ADMIN_ID]`, `[DEPT_ID]` | 관리자·학과장 리눅스 아이디 | |
| `[TOKEN_16]` | 서버 가입 토큰(16글자) | |

## 이 안내서의 규칙 (콘솔 작업 요령)
1. **서버 물리 콘솔에는 한글 글꼴이 없어** 한글이 `◆` 로 깨집니다. 스크립트 출력은 영어입니다. 가능하면 PC 에서 SSH 로 접속해 작업하세요(관리 서버가 사설망 안이면 서버실에서).
2. **한/영 키**가 영어 상태인지 확인하세요. 한글 상태로 치면 명령이 깨집니다(`ㄴㅕㅇㅐ` 처럼 보임).
3. 긴 명령은 틀리기 쉽습니다. **한 줄씩 나눠** 치고, 토큰·비밀번호는 명령에 직접 쓰지 말고 `read -r 변수` 로 입력받습니다.
4. `-o`(영문 오)와 `-i`(영문 아이), `/`와 `\`, `|` 를 구분하세요.
5. 서버 설정을 바꾸는 작업은 **되돌리는 방법**을 먼저 확인하고 합니다(각 노트북에 적혀 있음).
6. 비밀(토큰·비밀번호·인증 키)은 **채팅·사진·공개 저장소에 올리지 않습니다.**

## 전체 순서와 노트북

| 순서 | 노트북 | 내용 |
|---|---|---|
| 1 | [01_network_private_ip](01_network_private_ip.ipynb) | 공인 IP → 사설 IP 전환, DNS, 방화벽 |
| 2 | [02_bootstrap_mgmt](02_bootstrap_mgmt.ipynb) | 관리 서버에 K3s·ArgoCD·Sealed Secrets·Tailscale 설치 |
| 3 | [03_tailscale_funnel](03_tailscale_funnel.ipynb) | 학교 밖 접속(Funnel) |
| 4 | [04_secrets_deploy](04_secrets_deploy.ipynb) | 비밀값, 토큰, 이미지·저장소 접근, 사이트 배포 |
| 5 | [05_accounts](05_accounts.ipynb) | 학과장·관리자·학생 계정 |
| 6 | [06_join_servers](06_join_servers.ipynb) | 서버 등록(가입), 관리 서버 자신의 가입, 재가입 |
| 7 | [07_operations](07_operations.ipynb) | 웹 터미널·파일 탐색기, 백업, 만료, 강제 재시작, 배포 |
| 8 | [08_troubleshooting](08_troubleshooting.ipynb) | 실제로 겪은 문제와 해결 |

> **한 번에 한 서버씩**, 서버실에서 콘솔 앞에 앉아 진행하세요. 첫 서버는 **관리 서버**입니다.